# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [111]:
# import the requests library (1 line)
import requests

# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com"

# assign the /status endpoint to another variable called status_url (1 line)
status_url = f"{root_url}/status"

# query the /status endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(status_url)

# check the status_code using a condition and print appropriate messages (4 lines)
if req.status_code == 200:
    print(req.text)
else:
    print(req.status_code)

"Alive"


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [112]:
                                                    # Set the countries_url variable 
countries_url = f"{root_url}/countries"

                                                    # query the /countries endpoint using the get() method and store it in the req var
req = requests.get(countries_url)

                                                    # Get the json content and store it in the countries var
countries = req.json()
                                                    # display the request's status code and the countries var
print(req.status_code)
print(countries)

403
{'message': 'The cookie is missing'}


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [113]:
                                            # Set the cookie_url var
cookie_url = f"{root_url}/cookie"

                                            # query the cookie_url endpoint using the get() method and store it in the res var
res = requests.get(cookie_url)

                                            # assign the cookie from the response to a cookies var
cookies = res.cookies
                                            # display the cookies var
print(cookies)


<RequestsCookieJar[<Cookie user_cookie=df71d4fe-bb92-428d-9874-6cc74336887d for country-leaders.onrender.com/>]>


In [114]:
                                                             # Get a fresh cookie
cookie_url = f"{root_url}/cookie"
res = requests.get(cookie_url)
cookies = res.cookies

                                                            # Use it immediately to get the countries
countries_url = f"{root_url}/countries"
req = requests.get(countries_url, cookies=cookies)
countries = req.json()
                                                            # Display the results
print("Status Code:", req.status_code)
print("Countries:", countries)

Status Code: 200
Countries: ['fr', 'us', 'be', 'ma', 'ru']


Try to query the countries endpoint using the cookie, save the output and print it.

Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

In [115]:
                                            # refreshing cookie to avoid expiration
cookie_url = f"{root_url}/cookie"
cookies = requests.get(cookie_url).cookies

                                            # query the /leaders endpoint using cookies and params
leaders_url = f"{root_url}/leaders"
                                            # assign the output to the leaders variable
leaders = requests.get(leaders_url, cookies=cookies, params={"country": "fr"}).json()

                                            # display leaders var
print(leaders)

[{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_url': 'h

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [116]:
cookies = requests.get(f"{root_url}/cookie").cookies
                                                # Set leaders_url variable 
leaders_url = f"{root_url}/leaders"

                                                # query the /leaders endpoint, assign the output to the leaders var
leaders = requests.get(leaders_url, cookies=cookies).json()
                                                # display leaders var
print(leaders)

{'message': 'Please specify a country'}


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [117]:
                                                        # refresh cookie
cookies = requests.get(f"{root_url}/cookie").cookies
                                                        # query the /leaders endpoint using cookies and parameters (take any country in countries)
leaders = requests.get(leaders_url, cookies=cookies, params={"country": "us"}).json()

print(leaders)

[{'id': 'Q23', 'first_name': 'George', 'last_name': 'Washington', 'birth_date': '1732-02-22', 'death_date': '1799-12-14', 'place_of_birth': 'Westmoreland County', 'wikipedia_url': 'https://en.wikipedia.org/wiki/George_Washington', 'start_mandate': '1789-04-30', 'end_mandate': '1797-03-04'}, {'id': 'Q76', 'first_name': 'Barack', 'last_name': 'Obama', 'birth_date': '1961-08-04', 'death_date': None, 'place_of_birth': 'Kapiolani Medical Center for Women and Children', 'wikipedia_url': 'https://en.wikipedia.org/wiki/Barack_Obama', 'start_mandate': '2009-01-20', 'end_mandate': '2017-01-20'}, {'id': 'Q91', 'first_name': 'Abraham', 'last_name': 'Lincoln', 'birth_date': '1809-02-12', 'death_date': '1865-04-15', 'place_of_birth': 'Sinking Spring Farm', 'wikipedia_url': 'https://en.wikipedia.org/wiki/Abraham_Lincoln', 'start_mandate': '1861-03-04', 'end_mandate': '1865-04-15'}, {'id': 'Q207', 'first_name': 'George', 'last_name': 'Bush', 'birth_date': '1946-07-06', 'death_date': None, 'place_of_bi

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [118]:
cookies = requests.get(f"{root_url}/cookie").cookies
countries = requests.get(f"{root_url}/countries", cookies=cookies).json()

leaders_per_country = {}
for country in countries:
    cookies = requests.get(f"{root_url}/cookie").cookies
    leaders_per_country[country] = requests.get(leaders_url, cookies=cookies, params={"country": country}).json()

print(leaders_per_country.keys())

dict_keys(['fr', 'us', 'be', 'ma', 'ru'])


In [119]:
leaders_per_country = {country: requests.get(leaders_url, cookies=requests.get(f"{root_url}/cookie").cookies, 
    params={"country": country}).json() for country in countries}
print(leaders_per_country.keys())

dict_keys(['fr', 'us', 'be', 'ma', 'ru'])


It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [120]:
def get_leaders():
                                                    # define the urls, get the cookies, get the countries
    root_url = "https://country-leaders.onrender.com"
    countries_url, leaders_url = f"{root_url}/countries", f"{root_url}/leaders"
    countries = requests.get(countries_url, cookies=requests.get(f"{root_url}/cookie").cookies).json()
    
                                                    # loop over them and save leaders in a dict and return it
    return {c: requests.get(leaders_url, cookies=requests.get(f"{root_url}/cookie").cookies, params={"country": c}).json() for c in countries}


Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [121]:
leaders_per_country = get_leaders()
print(leaders_per_country.keys())

dict_keys(['fr', 'us', 'be', 'ma', 'ru'])


## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [122]:
wiki_url = "https://fr.wikipedia.org/wiki/Geoffrey_Hinton"
res = requests.get(wiki_url)
print(res.text[:1000])

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



In [123]:
headers = {"User-Agent": "SiegriedSandbox (siegca@hotmail.com)"}

res = requests.get(wiki_url, headers=headers)
print(res.text[:1000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="fr" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Geoffrey Hinton — Wikipédia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vect

Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [124]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(res.text, "html.parser")
print(soup.prettify()[:1000])


<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="fr">
 <head>
  <meta charset="utf-8"/>
  <title>
   Geoffrey Hinton — Wikipédia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-p

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [125]:
paragraphs = soup.find_all("p")
print(f"Nb of found paragraphs : {len(paragraphs)}")

Nb of found paragraphs : 36


In [126]:
first_paragraph = None
for p in paragraphs:
    cleaned_text = p.get_text().strip()
    if cleaned_text:                    # if text not empty
        first_paragraph = cleaned_text
        break                           # we stop when we've found the 1st
print(first_paragraph)

Pour les articles homonymes, voir Hinton.


In [127]:
first_paragraph = None
for p in paragraphs:
    text = p.get_text().strip()
    if text and len(text) > 50: 
        first_paragraph = text
        break
print(first_paragraph)

Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.


If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [128]:
                                            # Loop over the HTML paragraphs
for p in paragraphs:
    if p.find("b") and len(p.get_text().strip()) > 30:
                                            # Store the text
        first_paragraph = p.get_text().strip()
                                            # Exit
        break
print(first_paragraph)

Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.


At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [129]:
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)  
                                            # User-Agent definition
    headers = {"User-Agent": "SiegriedSandbox (siegca@hotmail.com)"}

    html_content = requests.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html_content, "html.parser")

    for p in soup.find_all("p"):
        if p.find("b") and len(p.get_text().strip()) > 30:
            first_paragraph = p.get_text().strip()
            break
    return first_paragraph

In [130]:
test_url = "https://fr.wikipedia.org/wiki/Geoffrey_Hinton"
output = get_first_paragraph(test_url)
print(output)

https://fr.wikipedia.org/wiki/Geoffrey_Hinton
Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.


### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [131]:
import re

                                                        # Define a sample text to clean
text_to_clean = "Geoffrey Hinton [1], born December 6, 1947 [2], is a computer scientist."
print("Before cleaning:", text_to_clean)

                                                        # regex to remove Wikipedia references like [1] or [2]
clean_paragraph = re.sub(r"\[.*?\]", "", text_to_clean)

                                                        # Display the final text
print("After cleaning :", clean_paragraph)

Before cleaning: Geoffrey Hinton [1], born December 6, 1947 [2], is a computer scientist.
After cleaning : Geoffrey Hinton , born December 6, 1947 , is a computer scientist.


In [132]:
# in all languages
import re
clean_paragraph = re.sub(r"\[.*?\]", "", first_paragraph).replace("  ", " ")
print(clean_paragraph)

Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.


Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [133]:
import re
from bs4 import BeautifulSoup

def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)  

    headers = {"User-Agent": "SiegriedSandbox (siegca@hotmail.com)"}
                                                        # Fetch the HTML content and parse
    html_content = requests.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html_content, "html.parser")
                                                        # Loop over the HTML paragraphs to find the 1st valid bio text
    for p in soup.find_all("p"):
        if p.find("b") and len(p.get_text().strip()) > 30:
            raw_paragraph = p.get_text().strip()
                                                        # regex to sanitize
            first_paragraph = re.sub(r"\[.*?\]", "", raw_paragraph)
            break
    return first_paragraph

In [134]:
# Test updated function with a Wikipedia URL
test_url = "https://fr.wikipedia.org/wiki/Geoffrey_Hinton"
pristine_text = get_first_paragraph(test_url)
print(pristine_text)

https://fr.wikipedia.org/wiki/Geoffrey_Hinton
Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.


Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [ ]:
import re
from bs4 import BeautifulSoup

def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)  

    headers = {"User-Agent": "SiegfriedSandbox (siegried@example.com)"}

                                                            # Fetch the HTML content and parse
    html_content = requests.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html_content, "html.parser")

    first_paragraph = ""

    for p in soup.find_all("p"):
        if p.find("b") and len(p.get_text().strip()) > 40:
            text = p.get_text()

                                                            # remove references/citations like [1], [a], [12]
            text = re.sub(r"\[.*?\]", "", text)

                                                            # Remove empty parentheses or ( ; ) or ( )
            text = re.sub(r"\(\s*([;,.\s]*)\s*\)", "", text)

                                                            # Clean up any lingering multiple spaces or tabs into a single space
            text = re.sub(r"\s+", " ", text)

                                                            # delete leading/trailing whitespace
            first_paragraph = text.strip()
            break
    return first_paragraph

In [136]:
# 1: French page 
fr_url = "https://fr.wikipedia.org/wiki/Geoffrey_Hinton"
print("FR Output:", get_first_paragraph(fr_url))
print("-" * 50)

# 2: English page
en_url = "https://en.wikipedia.org/wiki/Geoffrey_Hinton"
print("EN Output:", get_first_paragraph(en_url))

https://fr.wikipedia.org/wiki/Geoffrey_Hinton
FR Output: Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.
--------------------------------------------------
https://en.wikipedia.org/wiki/Geoffrey_Hinton
EN Output: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, cognitive psychologist and Nobel Prize laureate known for his work on artificial neural networks, which earned him the title "the Godfather of AI". He is University Professor Emeritus at the University of Toronto.


## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [137]:
def get_leaders():
                                                        # initial leaders list from API
    root_url = "https://country-leaders.onrender.com"   # Adjust if your API base URL is different
    cookie = get_cookie()                               # Reuses existing cookie management fct
    leaders_url = f"{root_url}/leaders"
    
    countries = requests.get(f"{root_url}/countries", cookies=cookie).json()
    leaders_data = []

    for country in countries:
        req = requests.get(leaders_url, params={"country": country}, cookies=cookie)
        leaders = req.json()
        
                                                        # inner loop over each to scrape intro
        for leader in leaders:
            wiki_url = leader.get("wikipedia_url")
            if wiki_url:
                                                        # Extract and clean the paragraph
                leader["biography"] = get_first_paragraph(wiki_url)
            else:
                leader["biography"] = "No Wikipedia URL provided."
                
            leaders_data.append(leader)
            
    return leaders_data

In [138]:
import requests

                                        # def of the get_cookie function
def get_cookie():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = f"{root_url}/cookie"
                                         # Request a fresh cookie from the API
    response = requests.get(cookie_url)
                                         # Return the cookies object directly so it can be passed to future requests
    return response.cookies

In [139]:
all_leaders = get_leaders()
print(all_leaders[0].get("biography")[:300] if all_leaders else "No data found")

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

AttributeError: 'str' object has no attribute 'get'

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [ ]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie = get_cookie()
    countries = requests.get(f"{root_url}/countries", cookies=cookie).json()
    leaders_data = []

    for country in countries:
        req = requests.get(f"{root_url}/leaders", params={"country": country}, cookies=cookie)

                                                # check if the cookies expired
        if req.status_code in [401, 403]:
            print(f"Cookies expired while fetching {country}! Refreshing...")
            cookie = get_cookie()               # fresh cookie
                                                # retry the request 
            req = requests.get(f"{root_url}/leaders", params={"country": country}, cookies=cookie,)

        leaders = req.json()
        for leader in leaders:
            if isinstance(leader, dict) and leader.get("wikipedia_url"):
                leader["biography"] = get_first_paragraph(leader["wikipedia_url"])
                leaders_data.append(leader)

    return leaders_data

Check the output of your function again.

In [ ]:
all_leaders = get_leaders()

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [140]:
import re
from bs4 import BeautifulSoup

def get_first_paragraph(wikipedia_url, session):
    print(wikipedia_url)  

    headers = {"User-Agent": "SiegriedSandboxProject/1.0 (siegca@hotmail.com)"}

                                                            # Use the shared session 
    html_content = session.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html_content, "html.parser")

    first_paragraph = ""
    for p in soup.find_all("p"):
        if p.find("b") and len(p.get_text().strip()) > 40:
            text = p.get_text()
            text = re.sub(r"\[.*?\]", "", text)
            text = re.sub(r"\(\s*([;,.\s]*)\s*\)", "", text)
            first_paragraph = re.sub(r"\s+", " ", text).strip()
            break

    return first_paragraph

In [141]:
import requests
# open session
with requests.Session() as my_session:
    
    test_url = "https://fr.wikipedia.org/wiki/Geoffrey_Hinton"
    resultat = get_first_paragraph(test_url, my_session)
    print(resultat)

https://fr.wikipedia.org/wiki/Geoffrey_Hinton
Geoffrey Everest Hinton, né le 6 décembre 1947 à Wimbledon (Royaume-Uni), est un chercheur britanno-canadien spécialiste de l'intelligence artificielle, de la psychologie cognitive à l'Université de Toronto et plus particulièrement des réseaux de neurones artificiels.


Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [ ]:
import requests

def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie = get_cookie()
    countries = requests.get(f"{root_url}/countries", cookies=cookie).json()
    leaders_data = []

    # Create a single Session object outside of the country loops for Wikipedia calls
    with requests.Session() as wiki_session:
        for country in countries:
            req = requests.get(f"{root_url}/leaders", params={"country": country},cookies=cookie,)

            if req.status_code in [401, 403]:
                cookie = get_cookie()
                req = requests.get(f"{root_url}/leaders", params={"country": country}, cookies=cookie,))

            leaders = req.json()
            for leader in leaders:
                if isinstance(leader, dict) and leader.get("wikipedia_url"):
                    # Pass the active session down as an argument
                    leader["biography"] = get_first_paragraph(leader["wikipedia_url"], wiki_session)                   leaders_data.append(leader)

    return leaders_data

Test your new functions.



In [143]:
all_leaders = get_leaders()

# confirm success (total structure gathered)
print(f"\nTotal leaders collected: {len(all_leaders)}")
if all_leaders:
    print(f"First leader entry biography preview:\n{all_leaders[0]['biography'][:200]}...")

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [144]:
import json
                                                    # save dataset to json file
with open("leaders.json", "w", encoding="utf-8") as f:
    json.dump(all_leaders, f, ensure_ascii=False, indent=4)

Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [145]:
                                                    # read file back into memory
with open("leaders.json", "r", encoding="utf-8") as f:
    loaded_leaders = json.load(f)

                                                    # check that the loaded data is identical to the original
print(f"Verification match: {all_leaders == loaded_leaders}")

Verification match: True


Make a function `save(leaders_per_country)` to call this code easily.

In [146]:
import json

def save(leaders_per_country):
    with open("leaders.json", "w", encoding="utf-8") as f:
        json.dump(leaders_per_country, f, ensure_ascii=False, indent=4)

In [147]:
save(all_leaders)
print(all_leaders)

[{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14', 'biography': "François Hollande Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017."}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15', 'biography': "Nicolas Sarközy de Nagy-Bocsa, dit Nicolas Sarkozy (/ni.kɔ.la saʁ.kɔ.zi/ Écouterⓘ ; en hongrois Sárközy ou Sárközi ,,), né le 28 janvier 1955 à Paris 17e (Seine), est un homme d'État français. Il est président de la République 

## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!